In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import umap
import hdbscan
import matplotlib.pyplot as plt

from tqdm import tqdm

In [ ]:
from collections import Counter

from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
cur_dir = os.getcwd().split('\\')

if cur_dir[-1] == 'notebooks':
    os.chdir("..")

from utils.data_load_utilities.data_loader import load_model_results
from utils.get_global_const import get_global_const
from utils.get_metrics import get_metrics
from utils.get_ranks import get_ranks, get_ranks_s
from methods.ADoE_method import *
from methods.k_nearest_methods import *
from methods.kmeans_methods import *
from methods.opt_methods import *
from methods.sparce_methods import *
from methods.entrophy_metods import *

In [ ]:
import tsfresh
import json

In [ ]:
SEED = 42
np.random.seed(SEED)

In [ ]:
results = []
_, datasets, _ = get_global_const()

In [ ]:
DATASETS_DIR = Path("data/datasets_metrics/datasets")
IN_DIR = Path("data/datasets_features")
OUT_DIR = Path("data/datasets_features")

In [ ]:
SEED = 42

In [ ]:
KNN_K = 1
TREE_DEPTH_SHALLOW = 3
CV_FOLDS = 5

In [ ]:
BOOT_N = 50
BOOT_FRAC = 0.8

In [ ]:
def load_dataset_json(dataset_name):
    
    with open(DATASETS_DIR / f"{dataset_name}.json", "r") as f:
        return json.load(f)

In [ ]:
def load_labels(time_data):

    if "y" in time_data:
        y = time_data["y"]
    elif "Y" in time_data:
        y = time_data["Y"]
    elif "labels" in time_data:
        y = time_data["labels"]
    else:
        raise KeyError("Could not find labels in dataset json. Expected one of: y, Y, labels.")
    
    return np.asarray(y)

In [ ]:
def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return float(default)

In [ ]:
def min_class_count(y):
    c = Counter(y.tolist())
    return int(min(c.values())) if len(c) > 0 else 0

def can_stratify(y, min_per_class=2):
    return min_class_count(y) >= min_per_class

def safe_train_test_split(X, y, test_size=0.3, random_state=0):

    if can_stratify(y, min_per_class=2):
        return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    else:
        return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=None)

In [ ]:
def make_cv_splitter(y, desired_folds, random_state=0):

    mcc = min_class_count(y)
    
    if mcc >= 2:
        n_splits = min(desired_folds, mcc)
        if n_splits >= 2:
            return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    n_splits = min(desired_folds, max(2, min(5, len(y))))
    
    return KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

In [ ]:
def linear_gap_train_test(X, y):

    Xtr, Xte, ytr, yte = safe_train_test_split(
        X, y, test_size=0.3, random_state=SEED
    )

    clf = make_pipeline(
        StandardScaler(with_mean=True, with_std=True),
        LogisticRegression(max_iter=2000)
    )

    if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
        return 0.0

    clf.fit(Xtr, ytr)
    acc_tr = accuracy_score(ytr, clf.predict(Xtr))
    acc_te = accuracy_score(yte, clf.predict(Xte))
    return float(acc_tr - acc_te)

In [ ]:
def cv_error_variance(model, X, y, desired_folds=5):

    splitter = make_cv_splitter(y, desired_folds, random_state=SEED)
    errs = []

    for tr_idx, te_idx in splitter.split(X, y):
        Xtr, Xte = X[tr_idx], X[te_idx]
        ytr, yte = y[tr_idx], y[te_idx]

        if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
            continue

        model.fit(Xtr, ytr)
        acc = accuracy_score(yte, model.predict(Xte))
        errs.append(1.0 - acc)

    if len(errs) < 2:
        return 0.0
    
    return float(np.var(errs, ddof=1))

In [ ]:
def bootstrap_error_and_rank_variance(models, X, y,
                                     n_boot=50, frac=0.8,
                                     max_resample_tries=10):

    n = X.shape[0]
    if n < 4:
        return 0.0, 0.0

    m_names = list(models.keys())
    target_name = "tree_shallow" if "tree_shallow" in models else m_names[0]
    target_idx = m_names.index(target_name)

    rng = np.random.default_rng(SEED)
    target_errors = []
    target_ranks = []

    for b in range(n_boot):

        idx = None
        for _ in range(max_resample_tries):
            cand = rng.choice(n, size=max(2, int(frac * n)), replace=True)
            yb = y[cand]
            if can_stratify(yb, min_per_class=2) or not can_stratify(y, min_per_class=2):
                idx = cand
                break
            
        if idx is None:
            idx = rng.choice(n, size=max(2, int(frac * n)), replace=True)

        Xb, yb = X[idx], y[idx]

        Xtr, Xte, ytr, yte = safe_train_test_split(
            Xb, yb, test_size=0.3, random_state=SEED + b
        )

        if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
            continue

        errs = []
        for name in m_names:
            mdl = models[name]
            mdl.fit(Xtr, ytr)
            acc = accuracy_score(yte, mdl.predict(Xte))
            errs.append(1.0 - acc)

        errs = np.asarray(errs, dtype=float)

        order = np.argsort(errs)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(order) + 1)

        target_errors.append(errs[target_idx])
        target_ranks.append(ranks[target_idx])

    if len(target_errors) < 2:
        return 0.0, 0.0

    var_err = float(np.var(target_errors, ddof=1))
    var_rank = float(np.var(target_ranks, ddof=1))
    
    return var_err, var_rank

In [ ]:
def compute_probe_errors(X, y):

    models = {
        "stump": DecisionTreeClassifier(max_depth=1, random_state=SEED),
        "knn1": KNeighborsClassifier(n_neighbors=KNN_K),
        "linear": make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            LogisticRegression(max_iter=2000)
        ),
        "tree_shallow": DecisionTreeClassifier(max_depth=TREE_DEPTH_SHALLOW, random_state=SEED),
    }

    splitter = make_cv_splitter(y, CV_FOLDS, random_state=SEED)
    cv_errs = {k: [] for k in models.keys()}

    for tr_idx, te_idx in splitter.split(X, y):
        
        Xtr, Xte = X[tr_idx], X[te_idx]
        ytr, yte = y[tr_idx], y[te_idx]

        if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
            continue

        for name, mdl in models.items():
            mdl.fit(Xtr, ytr)
            acc = accuracy_score(yte, mdl.predict(Xte))
            cv_errs[name].append(1.0 - acc)

    mean_err = {}
    
    for name, vals in cv_errs.items():
        mean_err[name] = float(np.mean(vals)) if len(vals) > 0 else 0.0

    return mean_err, models

In [ ]:
def dataset_meta_features_from_representation(
    df_rep,
    dataset_name):

    time_data = load_dataset_json(dataset_name)
    y = load_labels(time_data)

    df_ds = df_rep[df_rep["dataset"] == dataset_name].copy()
    df_ds = df_ds.sort_values("series_id")

    if len(df_ds) != len(y):
        raise ValueError(
            f"{dataset_name}: series count mismatch: rep rows={len(df_ds)} vs labels={len(y)}"
        )

    feature_cols = [c for c in df_ds.columns if c not in ("dataset", "series_id")]
    X = df_ds[feature_cols].to_numpy(dtype=float)

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    n_objects = int(X.shape[0])
    n_features = int(X.shape[1])

    series_len = None
    
    if "len" in df_ds.columns:
        series_len = float(np.mean(df_ds["len"].to_numpy()))
    else:
        try:
            first_series = np.asarray(time_data["X"][0][0])
            series_len = float(len(first_series))
        except Exception:
            series_len = np.nan

    sparsity_proxy = float(n_objects / max(1, n_features))

    mean_err, probe_models = compute_probe_errors(X, y)

    err_stump = mean_err["stump"]
    err_knn = mean_err["knn1"]
    err_linear = mean_err["linear"]
    err_tree = mean_err["tree_shallow"]

    gap_linear = linear_gap_train_test(X, y)

    diff_linear_tree = float(err_linear - err_tree)
    ratio_linear_knn = float(err_linear / (err_knn + 1e-12))

    var_cv_simple = cv_error_variance(
        DecisionTreeClassifier(max_depth=TREE_DEPTH_SHALLOW, random_state=SEED),
        X, y, desired_folds=CV_FOLDS
    )

    var_boot_err, var_boot_rank = bootstrap_error_and_rank_variance(
        probe_models, X, y, n_boot=BOOT_N, frac=BOOT_FRAC
    )

    return {
        "dataset": dataset_name,

        "err_stump": safe_float(err_stump),
        "err_knn1": safe_float(err_knn),
        "err_linear": safe_float(err_linear),
        "gap_linear_train_test": safe_float(gap_linear),
        "err_tree_shallow": safe_float(err_tree),

        "diff_err_linear_minus_tree": safe_float(diff_linear_tree),
        "ratio_err_linear_over_knn": safe_float(ratio_linear_knn),
        "var_err_cv_simple": safe_float(var_cv_simple),

        "var_err_boot_simple": safe_float(var_boot_err),
        "var_rank_boot_simple": safe_float(var_boot_rank),

        "n_objects": n_objects,
        "series_len_mean": safe_float(series_len),
        "n_features": n_features,
        "objects_per_feature": safe_float(sparsity_proxy),
    }

In [ ]:
def build_dataset_level_descriptors(
    rep_csv_path,
    out_csv_path,
    datasets,):
    
    df_rep = pd.read_csv(rep_csv_path)

    required = {"dataset", "series_id"}
    
    if not required.issubset(df_rep.columns):
        raise ValueError(f"{rep_csv_path} must contain columns: {required}")

    results = []
    
    for ds in datasets:
        print(f"[META] {rep_csv_path.name} -> {ds}")
        row = dataset_meta_features_from_representation(df_rep, ds)
        results.append(row)

    df_out = pd.DataFrame(results)
    df_out.to_csv(out_csv_path, index=False)
    print(f"Saved: {out_csv_path}  shape={df_out.shape}")
    
    return df_out

In [ ]:
tsfresh_csv = IN_DIR / "tsfresh_series_features.csv"
catch22_csv = IN_DIR / "catch22_series_features.csv"
summary_csv = IN_DIR / "summary_series_features.csv"
minirocket_csv = IN_DIR / "minirocket_series_features.csv"

out_tsfresh = OUT_DIR / "apriori_meta_tsfresh_probe.csv"
out_catch22 = OUT_DIR / "apriori_meta_catch22_probe.csv"
out_summary = OUT_DIR / "apriori_meta_summary_probe.csv"
out_minirocket = OUT_DIR / "apriori_meta_minirocket_probe.csv"

In [ ]:
# build_dataset_level_descriptors(tsfresh_csv, out_tsfresh, datasets)

In [ ]:
# build_dataset_level_descriptors(catch22_csv, out_catch22, datasets)

In [ ]:
# build_dataset_level_descriptors(summary_csv, out_summary, datasets)

In [ ]:
# build_dataset_level_descriptors(minirocket_csv, out_minirocket, datasets)

In [ ]:
def pad_or_trim(x, L):
    x = np.asarray(x, dtype=float)
    if len(x) >= L:
        return x[:L]
    pad_val = x[-1] if len(x) > 0 else 0.0
    return np.pad(x, (0, L - len(x)), constant_values=pad_val)

In [ ]:
def dataset_landmarking_raw(dataset_name):
    time_data = load_dataset_json(dataset_name)
    y = load_labels(time_data)

    series = [np.asarray(s[0], dtype=float) for s in time_data["X"]]

    L = max(len(s) for s in series)
    X = np.stack([pad_or_trim(s, L) for s in series], axis=0)

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    mean_err, probe_models = compute_probe_errors(X, y)

    err_stump  = mean_err["stump"]
    err_knn = mean_err["knn1"]
    err_linear = mean_err["linear"]
    err_tree = mean_err["tree_shallow"]

    gap_linear = linear_gap_train_test(X, y)
    diff_linear_tree = err_linear - err_tree
    ratio_linear_knn = err_linear / (err_knn + 1e-12)

    var_cv = cv_error_variance(
        DecisionTreeClassifier(max_depth=TREE_DEPTH_SHALLOW, random_state=SEED),
        X, y, desired_folds=CV_FOLDS
    )

    var_boot_err, var_boot_rank = bootstrap_error_and_rank_variance(
        probe_models, X, y, n_boot=BOOT_N, frac=BOOT_FRAC
    )

    return {
        "dataset": dataset_name,

        "err_stump": err_stump,
        "err_knn1": err_knn,
        "err_linear": err_linear,
        "err_tree_shallow": err_tree,

        "gap_linear_train_test": gap_linear,
        "diff_err_linear_minus_tree": diff_linear_tree,
        "ratio_err_linear_over_knn": ratio_linear_knn,

        "var_err_cv_simple": var_cv,
        "var_err_boot_simple": var_boot_err,
        "var_rank_boot_simple": var_boot_rank,

        "n_objects": int(X.shape[0]),
        "series_len_mean": float(L),
    }

In [ ]:
rows = []

for ds in datasets:
    print(f"[LANDMARK RAW] {ds}")
    rows.append(dataset_landmarking_raw(ds))

df_landmark = pd.DataFrame(rows)
df_landmark.to_csv("data/datasets_features/landmarking_raw.csv", index=False)

print(df_landmark.shape)